#03_gold_kpi_experiencia.sql

Gold - KPI experiência (SAC + Digital)

##Objetivo
- Entrega para BI experiência do cliente SAC + digital, por competência e recortes

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria tabela kpi_experiencia

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.gold.kpi_experiencia (
  competencia STRING,
  uf STRING,
  segmento_vinculo STRING,

  vidas_expostas BIGINT,

  qtd_protocolos BIGINT,
  protocolos_por_1000_vidas DOUBLE,
  nps_medio DOUBLE,
  csat_medio DOUBLE,
  qtd_sla_inconsistente BIGINT,

  qtd_eventos_app BIGINT,
  qtd_http_invalid BIGINT,
  http_invalid_rate DOUBLE,
  latencia_media DOUBLE,

  gold_build_ts TIMESTAMP
)
USING DELTA;

##Constroe dataset gold

In [0]:
INSERT OVERWRITE healthcare_dev.gold.kpi_experiencia
WITH base AS (
  SELECT
    competencia,
    uf,
    segmento_vinculo,
    beneficiario_id,
    qtd_protocolos,
    nps_medio,
    csat_medio,
    qtd_sla_inconsistente,
    qtd_eventos_app,
    qtd_http_invalid,
    latencia_media
  FROM healthcare_dev.gold.mart_member_month
),
agg AS (
  SELECT
    competencia,
    uf,
    segmento_vinculo,
    COUNT(DISTINCT beneficiario_id) AS vidas_expostas,
    SUM(COALESCE(qtd_protocolos,0)) AS qtd_protocolos,
    AVG(nps_medio) AS nps_medio,
    AVG(csat_medio) AS csat_medio,
    SUM(COALESCE(qtd_sla_inconsistente,0)) AS qtd_sla_inconsistente,
    SUM(COALESCE(qtd_eventos_app,0)) AS qtd_eventos_app,
    SUM(COALESCE(qtd_http_invalid,0)) AS qtd_http_invalid,
    AVG(latencia_media) AS latencia_media
  FROM base
  GROUP BY competencia, uf, segmento_vinculo
)
SELECT
  competencia,
  uf,
  segmento_vinculo,
  vidas_expostas,
  qtd_protocolos,
  CASE WHEN vidas_expostas=0 THEN 0.0 ELSE ROUND(1000.0 * qtd_protocolos / vidas_expostas, 4) END AS protocolos_por_1000_vidas,
  nps_medio,
  csat_medio,
  qtd_sla_inconsistente,
  qtd_eventos_app,
  qtd_http_invalid,
  CASE WHEN qtd_eventos_app=0 THEN 0.0 ELSE ROUND(100.0 * qtd_http_invalid / qtd_eventos_app, 4) END AS http_invalid_rate,
  latencia_media,
  current_timestamp() AS gold_build_ts
FROM agg;